# NB_02 — Electroplated Bi Process Window v4.1

**Engineering question**

> What electroplated-bismuth thickness and microstructure process window preserves Gaussian-like spectral response while increasing x-ray stopping power?

Version 4.1 keeps NB_02 as an orchestration layer over the reusable optimization modules while extending the evidence chain through SOURCE_04.

The notebook consumes SOURCE_01–04, the refreshed Engineering Objects, and the SYNTHESIS_01 handoff. It distinguishes reported operating points from validated tolerances, normalizes source-specific quantitative-value schemas for the reusable builder, and exports both process-window and specification artifacts.

Reusable logic lives in:

```text
tools/optimization/
    process_window_builder.py
    specification_builder.py
```

No numerical manufacturing tolerance is inferred from isolated literature operating points. Numerical candidate ranges remain unresolved until replicated process-to-response measurements support them.


## Workflow

```text
SYNTHESIS_01 handoff
        +
Engineering Objects
        +
SOURCE_01–SOURCE_04
        ↓
source-schema normalization
        ↓
process_window_builder.py
        ↓
reported operating points
quantitative evidence
candidate process-window dimensions
validation matrix
        ↓
specification_builder.py
        ↓
candidate specifications
open specifications
validation requirements
        ↓
NB_02 export
```


## 1. Locate repository and import the reusable process-window engine

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
    "SOURCE_03_electroplating_process.yaml",
    "SOURCE_04_thermal_conductivity.yaml",
]

NOTEBOOK_ID = "NB_02_ELECTROPLATED_BI_PROCESS_WINDOW"
PROCESS_WINDOW_ID = "PROCESS_WINDOW_01"
SYNTHESIS_ID = "SYNTHESIS_01"


def find_repo_root() -> Path:
    candidates = []
    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())
    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate
    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if target.exists():
            if (target / "engineering_navigator").is_dir():
                return target
            raise FileExistsError(f"{target} exists but does not look like sensors-becker.")
        subprocess.run(["git", "clone", REPOSITORY_URL, str(target)], check=True)
        if (target / "engineering_navigator").is_dir():
            return target
    raise FileNotFoundError("Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OBJ_DIR = ROOT / "engineering_navigator" / "engineering_objects"
SOURCE_DIR = (
    ROOT / "engineering_navigator" / "absorber_manufacturing" / "source_records"
)
SYNTHESIS_JSON = (
    ROOT / "outputs" / "engineering_questions" / "absorber_manufacturing"
    / SYNTHESIS_ID / "synthesis_summary.json"
)
OUTPUT_DIR = (
    ROOT / "outputs" / "engineering_questions" / "absorber_manufacturing"
    / PROCESS_WINDOW_ID
)
EXPORT_DIR = ROOT / "exports" / PROCESS_WINDOW_ID
EXPORT_ZIP = ROOT / "exports" / f"{PROCESS_WINDOW_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

from tools.optimization import (
    DEFAULT_ELECTROPLATED_BI_CONFIG,
    build_process_window,
    build_specification,
)

print(f"Repository : {ROOT}")
print("Imported   : tools.optimization process-window + specification builders")


## 2. Load SYNTHESIS_01 handoff, Engineering Objects, and completed source records


In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected one top-level YAML mapping")
    return data


if not SYNTHESIS_JSON.exists():
    raise FileNotFoundError(
        f"Missing synthesis handoff: {SYNTHESIS_JSON}. Run NB_01 v3.2 first."
    )

synthesis_summary = json.loads(SYNTHESIS_JSON.read_text(encoding="utf-8"))
handoff = synthesis_summary.get("next_notebook", {})
if handoff.get("id") != NOTEBOOK_ID:
    raise ValueError(
        f"SYNTHESIS_01 selects {handoff.get('id')!r}, expected {NOTEBOOK_ID!r}"
    )

expected_inputs = set(handoff.get("inputs", []))
if expected_inputs != set(SOURCE_FILES):
    raise ValueError(
        f"NB_02 source inputs differ from synthesis handoff: {sorted(expected_inputs)}"
    )

engineering_objects = {
    object_id: load_yaml(OBJ_DIR / f"{object_id}.yaml")
    for object_id in ("absorber", "electroplating", "tes")
}

records = {}
for filename in SOURCE_FILES:
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")
    if not source_id:
        raise KeyError(f"{filename}: missing source_id")
    if source_id in records:
        raise ValueError(f"Duplicate source_id: {source_id}")
    records[source_id] = record

incomplete = {
    source_id: record.get("extraction_status")
    for source_id, record in records.items()
    if not str(record.get("extraction_status", "")).startswith("complete")
}
if incomplete:
    raise ValueError("Incomplete source records: " + json.dumps(incomplete, indent=2))

print("SYNTHESIS_01 handoff: PASS")
print("Engineering Objects:", ", ".join(engineering_objects))
print("Source-record validation: PASS")
print("Sources:", ", ".join(sorted(records)))
print("Reported process points in synthesis:", synthesis_summary.get("reported_process_point_count"))


## 3. Normalize source-specific quantitative-value schemas

The reusable process-window builder expects `reported_values` records with `variable` and `value`. SOURCE_04 also uses `id`, `value_range`, and `values`. This cell builds an in-memory normalized copy for the builder without changing the canonical source YAML files.


In [ ]:
builder_records = deepcopy(records)

VARIABLE_ALIASES = {
    "electroplating_current_density": "current_density",
    "current_density": "current_density",
    "plating_rate": "plating_rate",
    "Bi_thickness": "Bi_thickness",
    "bismuth_thickness": "Bi_thickness",
}

for source_id, record in builder_records.items():
    normalized_values = []
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue
        normalized = dict(item)
        raw_variable = normalized.get("variable", normalized.get("id"))
        if raw_variable:
            normalized["variable"] = VARIABLE_ALIASES.get(raw_variable, raw_variable)

        if normalized.get("value") is None:
            if normalized.get("value_range") is not None:
                value_range = normalized["value_range"]
                if isinstance(value_range, (list, tuple)) and len(value_range) == 2:
                    normalized["value"] = f"{value_range[0]}-{value_range[1]}"
                else:
                    normalized["value"] = str(value_range)
            elif normalized.get("values") is not None:
                normalized["value"] = json.dumps(normalized["values"], ensure_ascii=False)

        normalized_values.append(normalized)

    record["reported_values"] = normalized_values

print("Source-schema normalization: PASS")


## 4. Build the process-window and specification packages

In [ ]:
result = build_process_window(
    engineering_objects=engineering_objects,
    source_records=builder_records,
    config=DEFAULT_ELECTROPLATED_BI_CONFIG,
)
specification = build_specification(result)

print("Process-window build: PASS")
print("Specification build : PASS")
print(f"Process variables        : {len(result.process_variables)}")
print(f"Quantitative evidence    : {len(result.quantitative_evidence)}")
print(f"Operating points         : {len(result.operating_points)}")
print(f"Coupled variables        : {len(result.coupled_variables)}")
print(f"Window dimensions        : {len(result.candidate_process_window)}")
print(f"Validation experiments   : {len(result.validation_matrix)}")
print(f"Candidate specifications : {len(specification.candidate_specifications)}")
print(f"Open specifications      : {len(specification.open_specifications)}")


## 5. Process variables

In [ ]:
result.process_variables


## 6. Quantitative evidence

In [ ]:
result.quantitative_evidence


## 7. Source-supported operating points

In [ ]:
if result.operating_points.empty:
    print("No source-supported process operating points recorded.")
else:
    result.operating_points


## 8. Coupled Engineering Object variables

In [ ]:
result.coupled_variables


## 9. Candidate process-window dimensions

A blank `candidate_range` is intentional. It means the repository does not yet contain enough replicated evidence to justify a numerical manufacturing tolerance.


In [ ]:
result.candidate_process_window


## 10. Validation matrix

In [ ]:
result.validation_matrix


## 11. Process-window status

In [ ]:
process_window_status = {
    "notebook_id": NOTEBOOK_ID,
    **result.status,
}

process_window_status


## 12. Engineering interpretation

The current evidence establishes **reported operating points**, not a validated numerical process window.

- SOURCE_01 reports an electroplated-Bi sample at 6.0 mA/cm² with 1.4 V bias and 283 nm/min plating rate.
- SOURCE_03 reports a high-quality film condition at 9 mA/cm² and 40 °C, with a measured thickness/grain-size point.
- SOURCE_04 reports a test-structure condition at 7.7 mA/cm², room temperature, pH 0.15, with an estimated 220–250 nm/min plating rate, and independently supports adequate absorber thermal conductivity.

These points delimit observed literature conditions but do **not** establish allowable intervals. The candidate dimensions therefore remain unresolved until controlled sweeps and replicated batches connect process inputs to grain-size distributions, spectral tailing, thermalization, yield, and repeatability.


## 13. Candidate specifications


In [ ]:
specification.candidate_specifications


## 14. Open specifications


In [ ]:
specification.open_specifications


## 15. Validation requirements


In [ ]:
specification.validation_requirements


## 16. Write outputs

In [ ]:
process_variables_csv = OUTPUT_DIR / "process_variables.csv"
quantitative_evidence_csv = OUTPUT_DIR / "quantitative_evidence.csv"
operating_points_csv = OUTPUT_DIR / "operating_points.csv"
coupled_variables_csv = OUTPUT_DIR / "coupled_variables.csv"
window_csv = OUTPUT_DIR / "candidate_process_window.csv"
validation_csv = OUTPUT_DIR / "validation_matrix.csv"
candidate_specs_csv = OUTPUT_DIR / "candidate_specifications.csv"
open_specs_csv = OUTPUT_DIR / "open_specifications.csv"
validation_requirements_csv = OUTPUT_DIR / "validation_requirements.csv"
status_json = OUTPUT_DIR / "process_window_status.json"
spec_status_json = OUTPUT_DIR / "specification_status.json"

result.process_variables.to_csv(process_variables_csv, index=False)
result.quantitative_evidence.to_csv(quantitative_evidence_csv, index=False)
result.operating_points.to_csv(operating_points_csv, index=False)
result.coupled_variables.to_csv(coupled_variables_csv, index=False)
result.candidate_process_window.to_csv(window_csv, index=False)
result.validation_matrix.to_csv(validation_csv, index=False)
specification.candidate_specifications.to_csv(candidate_specs_csv, index=False)
specification.open_specifications.to_csv(open_specs_csv, index=False)
specification.validation_requirements.to_csv(validation_requirements_csv, index=False)

status_json.write_text(json.dumps(process_window_status, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
spec_status_json.write_text(json.dumps(specification.status, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

written_files = {
    "process_variables": process_variables_csv,
    "quantitative_evidence": quantitative_evidence_csv,
    "operating_points": operating_points_csv,
    "coupled_variables": coupled_variables_csv,
    "candidate_process_window": window_csv,
    "validation_matrix": validation_csv,
    "candidate_specifications": candidate_specs_csv,
    "open_specifications": open_specs_csv,
    "validation_requirements": validation_requirements_csv,
    "process_window_status": status_json,
    "specification_status": spec_status_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(ROOT)}")


## 17. Build and download export ZIP

In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 18. Handoff

Version 4.1 closes the SOURCE_01–04 process-window handoff without promoting isolated reported points into manufacturing tolerances.

Expected current state:

- three source-supported electroplating operating points;
- unresolved numerical candidate ranges;
- candidate/open specifications generated from the reusable optimization layer;
- a validation matrix for thickness, current density, bias voltage, plating rate, grain size, and replicated batches;
- SOURCE_04 thermal-conductivity evidence retained as a detector-thermalization constraint rather than misread as a process tolerance.

The next substantive advance requires replicated process-to-response evidence. Until then, the notebook should remain in `measurement_plan_ready` state.

*Admissible generalizations trail leading specifications.*
